In [1]:
# Install dependencies
!pip install rank-bm25 sentence-transformers numpy --quiet

In [2]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

# Corpus

In [3]:
corpus = [
    "Transformers use self-attention mechanisms to process sequences in parallel.",
    "BERT is a bidirectional encoder trained using masked language modeling.",
    "The BM25 algorithm ranks documents based on term frequency and inverse document frequency.",
    "Gradient descent is an optimization algorithm used to minimize loss functions.",
    "Neural networks learn by adjusting weights through backpropagation.",
    "Adam optimizer improves gradient descent using momentum and adaptive learning rates.",
    "Attention mechanisms allow models to focus on relevant parts of the input sequence.",
    "Overfitting occurs when a model memorizes training data instead of generalizing.",
    "Regularization techniques like dropout help prevent overfitting in neural networks.",
    "XR-7700-B is a proprietary AI accelerator chip optimized for transformer workloads.",
    "Loss functions measure how far model predictions are from actual targets.",
]

print("Corpus size:", len(corpus))

Corpus size: 11


# Naive RAG (baseline)

In [4]:
sbert_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
corpus_embeddings = sbert_model.encode(corpus, convert_to_numpy=True)

def naive_retrieve(query, top_k=1):
    q_vec = sbert_model.encode([query], convert_to_numpy=True)[0]

    scores = corpus_embeddings @ q_vec / (
        np.linalg.norm(corpus_embeddings, axis=1) * np.linalg.norm(q_vec)
    )

    top_idx = np.argsort(scores)[::-1][:top_k]
    return [corpus[i] for i in top_idx]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# HybridRetriever

In [5]:
class HybridRetriever:
    def __init__(self, corpus, k=60):
        self.corpus = corpus
        self.k = k

        # BM25
        self.tokenized_corpus = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(self.tokenized_corpus)

        # SBERT
        self.sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        self.doc_embeddings = self.sbert.encode(corpus, convert_to_numpy=True)

    def retrieve(self, query, top_k=5):
        # BM25
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        bm25_ranked = np.argsort(bm25_scores)[::-1]
        bm25_ranks = {doc_id: rank+1 for rank, doc_id in enumerate(bm25_ranked)}

        # SBERT
        q_vec = self.sbert.encode([query], convert_to_numpy=True)[0]
        sbert_scores = self.doc_embeddings @ q_vec / (
            np.linalg.norm(self.doc_embeddings, axis=1) * np.linalg.norm(q_vec)
        )
        sbert_ranked = np.argsort(sbert_scores)[::-1]
        sbert_ranks = {doc_id: rank+1 for rank, doc_id in enumerate(sbert_ranked)}

        # RRF
        scores = {}
        for doc_id in range(len(self.corpus)):
            scores[doc_id] = (
                1 / (self.k + bm25_ranks[doc_id]) +
                1 / (self.k + sbert_ranks[doc_id])
            )

        ranked_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

        results = []
        for doc_id, score in ranked_docs:
            results.append({
                "doc_id": doc_id,
                "rrf_score": score,
                "bm25_rank": bm25_ranks[doc_id],
                "sbert_rank": sbert_ranks[doc_id],
                "text": self.corpus[doc_id]
            })

        return results

# Re-ranker

In [6]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, candidates, top_k=3):
    pairs = [(query, doc["text"]) for doc in candidates]
    scores = cross_encoder.predict(pairs)

    for i, doc in enumerate(candidates):
        doc["cross_score"] = float(scores[i])

    reranked = sorted(candidates, key=lambda x: x["cross_score"], reverse=True)
    return reranked[:top_k]


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

# Query expansion

In [7]:
def hyde_query(query):
    return f"""
    This query relates to machine learning concepts.
    Provide a detailed technical explanation involving:
    - optimization methods (gradient descent, Adam)
    - neural network training
    - attention mechanisms
    - relevant algorithms

    Query: {query}
    """

# Full pipeline

In [8]:
retriever = HybridRetriever(corpus)

def advanced_rag(user_query):
    # 1. Query Expansion
    expanded_query = hyde_query(user_query)

    # 2. Hybrid Retrieval
    candidates = retriever.retrieve(expanded_query, top_k=8)

    # 3. Re-ranking
    reranked = rerank(user_query, candidates, top_k=2)

    # 4. Return best document (no LLM needed)
    return reranked[0]["text"]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Comparison

In [9]:
queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is BM25 formula?"
]

results = []

for q in queries:
    naive = naive_retrieve(q)[0]
    advanced = advanced_rag(q)

    print("\n==============================")
    print("Query:", q)
    print("Naive Top Doc:", naive)
    print("Advanced Top Doc:", advanced)

    results.append((q, naive, advanced))


Query: how do transformers encode meaning?
Naive Top Doc: Transformers use self-attention mechanisms to process sequences in parallel.
Advanced Top Doc: Transformers use self-attention mechanisms to process sequences in parallel.

Query: optimization techniques for training
Naive Top Doc: Neural networks learn by adjusting weights through backpropagation.
Advanced Top Doc: Gradient descent is an optimization algorithm used to minimize loss functions.

Query: what is BM25 formula?
Naive Top Doc: The BM25 algorithm ranks documents based on term frequency and inverse document frequency.
Advanced Top Doc: The BM25 algorithm ranks documents based on term frequency and inverse document frequency.


## Comparison Table

| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Are they different? |
|---|---|---|---|
| how do transformers encode meaning? | Transformers use self-attention mechanisms... | Transformers use self-attention mechanisms... | No |
| optimization techniques for training | Neural networks learn by adjusting weights... | Gradient descent is an optimization algorithm... | Yes |
| what is BM25 formula? | The BM25 algorithm ranks documents... | The BM25 algorithm ranks documents... | No |

### Observations

- Advanced RAG improves retrieval for ambiguous queries, as seen in the optimization query where it correctly retrieves gradient descent instead of a loosely related concept.
- For queries where dense retrieval already performs well, Advanced RAG preserves correct results rather than altering them unnecessarily.
- Hybrid retrieval (BM25 + SBERT) improves recall by combining keyword matching with semantic understanding.
- Cross-encoder re-ranking ensures that the most contextually relevant document is selected from retrieved candidates.
- Overall, Advanced RAG is more robust and reliable compared to naïve dense retrieval.

# BONUS 1 — Weighted RRF

In [10]:
class WeightedHybridRetriever(HybridRetriever):
    def __init__(self, corpus, k=60, alpha=0.5):
        super().__init__(corpus, k)
        self.alpha = alpha

    def retrieve(self, query, top_k=5):
        # BM25
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        bm25_ranked = np.argsort(bm25_scores)[::-1]
        bm25_ranks = {doc_id: rank+1 for rank, doc_id in enumerate(bm25_ranked)}

        # SBERT
        q_vec = self.sbert.encode([query], convert_to_numpy=True)[0]
        sbert_scores = self.doc_embeddings @ q_vec / (
            np.linalg.norm(self.doc_embeddings, axis=1) * np.linalg.norm(q_vec)
        )
        sbert_ranked = np.argsort(sbert_scores)[::-1]
        sbert_ranks = {doc_id: rank+1 for rank, doc_id in enumerate(sbert_ranked)}

        # Weighted RRF
        scores = {}
        for doc_id in range(len(self.corpus)):
            scores[doc_id] = (
                self.alpha * (1 / (self.k + bm25_ranks[doc_id])) +
                (1 - self.alpha) * (1 / (self.k + sbert_ranks[doc_id]))
            )

        ranked_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

        return [self.corpus[doc_id] for doc_id, _ in ranked_docs]

In [11]:
alphas = [0.3, 0.5, 0.7]
query = "BM25 term frequency"

for a in alphas:
    retriever_w = WeightedHybridRetriever(corpus, alpha=a)
    result = retriever_w.retrieve(query, top_k=1)[0]

    print(f"\nAlpha = {a}")
    print("Top Doc:", result)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Alpha = 0.3
Top Doc: The BM25 algorithm ranks documents based on term frequency and inverse document frequency.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Alpha = 0.5
Top Doc: The BM25 algorithm ranks documents based on term frequency and inverse document frequency.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Alpha = 0.7
Top Doc: The BM25 algorithm ranks documents based on term frequency and inverse document frequency.


### Bonus 1 — Weighted RRF Observations

- Lower α (0.3) favors semantic retrieval (SBERT).
- Higher α (0.7) favors keyword-based retrieval (BM25).
- Keyword-heavy queries (e.g., BM25) perform better with higher α.
- Semantic queries perform better with lower α.
- α = 0.5 provides a balanced trade-off.

# BONUS 2 — Chunk Size Study

In [12]:
long_doc = """
Transformers are deep learning models that use self-attention mechanisms to process sequences.
They have revolutionized natural language processing tasks such as translation, summarization,
and question answering. Training involves optimization techniques such as gradient descent,
Adam optimizer, and learning rate scheduling. Regularization techniques like dropout prevent
overfitting. Attention allows models to focus on important tokens in a sequence.
""" * 10  # make it long (>500 words)


def chunk_text(text, chunk_size):
    words = text.split()
    return [
        " ".join(words[i:i+chunk_size])
        for i in range(0, len(words), chunk_size)
    ]

In [13]:
sizes = [50, 100, 200]
query = "how do transformers learn?"

for size in sizes:
    chunks = chunk_text(long_doc, size)

    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embeddings = model.encode(chunks, convert_to_numpy=True)

    q_vec = model.encode([query], convert_to_numpy=True)[0]

    scores = embeddings @ q_vec / (
        np.linalg.norm(embeddings, axis=1) * np.linalg.norm(q_vec)
    )

    best_chunk = chunks[np.argmax(scores)]

    print(f"\nChunk size: {size}")
    print("Top chunk preview:", best_chunk[:150], "...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Chunk size: 50
Top chunk preview: Transformers are deep learning models that use self-attention mechanisms to process sequences. They have revolutionized natural language processing ta ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Chunk size: 100
Top chunk preview: Transformers are deep learning models that use self-attention mechanisms to process sequences. They have revolutionized natural language processing ta ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Chunk size: 200
Top chunk preview: Transformers are deep learning models that use self-attention mechanisms to process sequences. They have revolutionized natural language processing ta ...


### Bonus 2 — Chunk Size Observations

- Smaller chunks (50 words) provide more precise retrieval but may lose context.
- Medium chunks (100 words) balance context and precision.
- Larger chunks (200 words) provide more context but may dilute relevance.
- Chunk size significantly affects retrieval quality in RAG systems.

# BONUS 3 — ColBERT Integration

In [14]:
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def colbert_score(query, document):
    q_tokens = query.lower().split()
    d_tokens = document.lower().split()

    q_vecs = encoder.encode(q_tokens, convert_to_numpy=True)
    d_vecs = encoder.encode(d_tokens, convert_to_numpy=True)

    q_vecs = q_vecs / np.linalg.norm(q_vecs, axis=1, keepdims=True)
    d_vecs = d_vecs / np.linalg.norm(d_vecs, axis=1, keepdims=True)

    sim_matrix = q_vecs @ d_vecs.T
    max_sims = sim_matrix.max(axis=1)

    return float(max_sims.sum())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
def hybrid_with_colbert(query, corpus, k=60, top_k=3):
    # BM25
    tokenized_corpus = [doc.lower().split() for doc in corpus]
    bm25 = BM25Okapi(tokenized_corpus)
    bm25_scores = bm25.get_scores(query.lower().split())
    bm25_ranked = np.argsort(bm25_scores)[::-1]
    bm25_ranks = {doc_id: rank+1 for rank, doc_id in enumerate(bm25_ranked)}

    # SBERT
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    doc_vecs = model.encode(corpus, convert_to_numpy=True)
    q_vec = model.encode([query], convert_to_numpy=True)[0]

    sbert_scores = doc_vecs @ q_vec / (
        np.linalg.norm(doc_vecs, axis=1) * np.linalg.norm(q_vec)
    )
    sbert_ranked = np.argsort(sbert_scores)[::-1]
    sbert_ranks = {doc_id: rank+1 for rank, doc_id in enumerate(sbert_ranked)}

    # ColBERT
    colbert_scores = [colbert_score(query, doc) for doc in corpus]
    colbert_ranked = np.argsort(colbert_scores)[::-1]
    colbert_ranks = {doc_id: rank+1 for rank, doc_id in enumerate(colbert_ranked)}

    # RRF (3 sources)
    scores = {}
    for doc_id in range(len(corpus)):
        scores[doc_id] = (
            1/(k + bm25_ranks[doc_id]) +
            1/(k + sbert_ranks[doc_id]) +
            1/(k + colbert_ranks[doc_id])
        )

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    return [corpus[doc_id] for doc_id, _ in ranked]

In [16]:
query = "how do neural networks learn?"

results = hybrid_with_colbert(query, corpus)

print("Top results with ColBERT fusion:")
for r in results:
    print("-", r)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Top results with ColBERT fusion:
- Neural networks learn by adjusting weights through backpropagation.
- Regularization techniques like dropout help prevent overfitting in neural networks.
- Loss functions measure how far model predictions are from actual targets.


### Bonus 3 — ColBERT Observations

- ColBERT improves fine-grained matching using token-level interactions.
- It captures semantic similarity better than SBERT alone.
- Combining BM25, SBERT, and ColBERT improves overall retrieval robustness.
- However, ColBERT is computationally more expensive.